# LangGraph Fundamentals

A collection of beginner-friendly LangGraph examples covering the fundamentals of building stateful LLM workflows.

## Topics Covered

* **Sequential Workflows** — Execute nodes in a fixed sequence.
* **Conditional Edges** — Dynamically route execution based on state.
* **State Management** — Define and update shared graph state using `TypedDict`.
* **LLM Integration** — Invoke an LLM from LangGraph nodes.
* **Checkpointing** — Save and retrieve graph state using `MemorySaver`.
* **SQLite Persistence** — Persist checkpoints using `SqliteSaver`.
* **Thread-Based Memory** — Maintain separate conversation histories using `thread_id`.
* **Message State** — Maintain conversation history using `add_messages`.

## Mental Flow

```text
User Input
    ↓
State
    ↓
Node
    ↓
Edge / Routing
    ↓
Next Node
    ↓
Updated State
    ↓
Checkpoint / Thread Memory
    ↓
LLM / Final Output
```

## Learning Progression

```text
Sequential Graph
      ↓
Conditional Routing
      ↓
State Management
      ↓
LLM Integration
      ↓
MemorySaver
      ↓
SQLite Persistence
      ↓
Thread-Based Memory
```


## LangGraph Basic Sequential Workflow

This example demonstrates a simple sequential LangGraph workflow where each node executes one after another.
`Node1` modifies the student's name, `Node2` increases the roll number, and `Node3` uses the updated state to generate a summary using an LLM.
The flow is `START → Node1 → Node2 → Node3 → END`, with each node receiving the state and returning only the fields it wants to update.

In [ ]:
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
import json

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

class StudentInfo(TypedDict):
    name: str
    roll: int
    summary: str


def call_student(state: StudentInfo):
    return {
        "name": state["name"] + " can bench 100kg and loves drinking black coffee"
    }


def rankno(state: StudentInfo):
    return {
        "roll": state["roll"] + 5
    }

def call_llm(state: StudentInfo):
    prompt=f"""This is name of the student{state["name"]} and this is his roll no {state["roll"]}
    desricbe him
    """
    response= llm.invoke(prompt)
    print(response.usage_metadata)
    return {"summary":response.content}
# Build graph
mah_graph = StateGraph(StudentInfo)

mah_graph.add_node("Node1", call_student)
mah_graph.add_node("Node2", rankno)
mah_graph.add_node("Node3", call_llm)

mah_graph.add_edge(START, "Node1")
mah_graph.add_edge("Node1", "Node2")



mah_graph.add_edge("Node2", "Node3")
mah_graph.add_edge("Node3", END)

app = mah_graph.compile()


# Run
result = app.invoke({
    "name": "Raj",
    "roll": 41,
    "summary": ""
})

print(json.dumps(result, indent=2, ensure_ascii=False))

{'input_tokens': 104, 'output_tokens': 247, 'total_tokens': 351, 'output_token_details': {'reasoning': 58}}
{
  "name": "Raj can bench 100kg and loves drinking black coffee",
  "roll": 46,
  "summary": "**Student Profile: Raj**\n\n- **Name:** Raj  \n- **Roll Number:** 46  \n- **Physical Strength:** Able to bench‑press 100 kg, indicating a solid commitment to fitness and a strong, athletic build.  \n- **Personal Preference:** Enjoys drinking black coffee, suggesting he appreciates a straightforward, no‑frills caffeine boost—perhaps to stay alert for studies or workouts.  \n\n**Overall Snapshot:**  \nRaj is a disciplined individual who balances academic life with a serious fitness routine. His ability to bench 100 kg reflects dedication to strength training, while his fondness for black coffee hints at a pragmatic, focused mindset—he likely values efficiency and enjoys the pure taste of coffee without added sugars or creams. As a student with roll number 46, he probably stands out among 

In [ ]:
mah_graph

## LangGraph Conditional Edges

This example demonstrates how conditional edges control the flow of a LangGraph workflow.  
After `Node1`, the `route_func()` checks the student's roll number and decides which node should execute next.  
If `roll > 5`, the graph moves to `Node2`; otherwise, it moves directly to `Node3`.  
The routing function returns the name of the next node, which LangGraph uses to determine the execution path.

In [ ]:
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
import json

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

class StudentInfo(TypedDict):
    name: str
    roll: int
    summary: str


def call_student(state: StudentInfo):
    return {
        "name": state["name"] + "can bench 100kg and loves drinking black coffee"
    }


def rankno(state: StudentInfo):
    return {
        "roll": state["roll"] + 5
    }

def call_llm(state: StudentInfo):
    prompt=f"""This is name of the student{state["name"]} and this is his roll no {state["roll"]}
    desricbe him
    """
    response= llm.invoke(prompt)
    return {"summary":response.content}

def route_func(state:StudentInfo):
    if state["roll"]>5:
        return "Node2"
    return "Node3"


# Build graph
mah_graph = StateGraph(StudentInfo)

mah_graph.add_node("Node1", call_student)
mah_graph.add_node("Node2", rankno)
mah_graph.add_node("Node3", call_llm)

mah_graph.add_edge(START, "Node1")
mah_graph.add_conditional_edges(
    "Node1",
    route_func
)



mah_graph.add_edge("Node2", "Node3")
mah_graph.add_edge("Node3", END)

app = mah_graph.compile()


# Run
result = app.invoke({
    "name": "Raj",
    "roll": 1,
    "summary": ""
})

print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "name": "Rajcan bench 100kg and loves drinking black coffee",
  "roll": 1,
  "summary": "**Student Profile: Rajcan (Roll No. 1)**  \n\n- **Name:** Rajcan  \n- **Roll Number:** 1  \n- **Physical Fitness:** Capable of bench‑pressing 100 kg, indicating strong upper‑body strength and a dedication to regular strength training.  \n- **Personal Taste:** A true coffee enthusiast who prefers his brew black—no sugar, no milk—suggesting he enjoys the pure, bold flavor of coffee and likely values the quick, focused energy boost it provides.  \n- **Overall Snapshot:** Rajcan combines academic commitment (as reflected by his prominent roll number) with a disciplined fitness routine and a simple, classic taste in beverages. This blend of intellect, physical strength, and straightforward preferences paints the picture of a well‑rounded, motivated student."
}


In [ ]:
mah_graph

## LangGraph Checkpointing with MemorySaver

This example uses `MemorySaver` to persist the graph's state using a `thread_id`.  
The same `config` can be used with `invoke()` to run the graph and `get_state()` to retrieve the saved state.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict


# State shared between all nodes
class State(TypedDict):
    name: str
    count: int
    message: str


# Node 1: Increment count
def add_one(state: State):
    return {
        "count": state["count"] + 1
    }


# Node 2: Create a message using the updated state
def create_message(state: State):
    return {
        "message": f"Hello {state['name']}, your count is {state['count']}"
    }


# Build graph
builder = StateGraph(State)

builder.add_node("add", add_one)
builder.add_node("message", create_message)

builder.add_edge(START, "add")
builder.add_edge("add", "message")
builder.add_edge("message", END)






# Checkpointer = memory
memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

config_0= {"thread_id":1}
config_1= {"thread_id":"1"}
# Conversation / state ID
config = {
    "configurable": {
        "thread_id": "1"
    }
}


# First run
result = graph.invoke(
    {
        "name": "Ashutosh",
        "count": 5,
        "message": ""
    },
    config
)

print(result)


# Retrieve saved state
saved_state = graph.get_state(config)

print(saved_state.values)

{'name': 'Ashutosh', 'count': 6, 'message': 'Hello Ashutosh, your count is 6'}
{'name': 'Ashutosh', 'count': 6, 'message': 'Hello Ashutosh, your count is 6'}


## LangGraph Persistent Chat Memory with SQLite

This example builds a chatbot that uses `SqliteSaver` to persist conversation state in a SQLite database.  
The `messages` state uses `add_messages`, allowing new messages to be appended to the existing conversation history.  
Each conversation is identified by a unique `thread_id`, so different threads maintain separate conversation histories.  
Here, `thread_id="1"` remembers that the user's name is Raj, while `thread_id="2"` has a separate conversation with Rana Sanga.  
When the second message is sent using `config1`, the chatbot retrieves the previous state from SQLite and can answer **"What is my name?"** using the history from thread 1.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import sqlite3


load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b"
)


# State
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


# Node
def chat_node(state: ChatState):

    messages = state["messages"]

    response = llm.invoke(messages)

    return {
        "messages": [response]
    }


# SQLite database
conn = sqlite3.connect(
    database="chatbot.db",
    check_same_thread=False
)

# Checkpointer
checkpointer = SqliteSaver(conn=conn)


# Build graph
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)


# Compile
chatbot = graph.compile(
    checkpointer=checkpointer
)


# =====================================================
# THREAD 1
# =====================================================

config1 = {
    "configurable": {
        "thread_id": "1"
    }
}
config2 = {
    "configurable": {
        "thread_id": "2"
    }
}


# First message
response1 = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="Hi, my name is Raj")
        ]
    },
    config1
)

# First message
response2 = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="Hi, my name is Rana Sanga")
        ]
    },
    config2
)



# Second message - SAME THREAD
response = chatbot.invoke(
    {
        "messages": [
            HumanMessage(content="What is my name?")
        ]
    },
    config1
)


print(response["messages"][-1].content)




Your name is Raj.


In [ ]:
print("\nTHREAD 1 STATE:")
state1 = chatbot.get_state(config1)

for message in state1.values["messages"]:
    print(f"{type(message).__name__}: {message.content}")


print("\nTHREAD 2 STATE:")
state2 = chatbot.get_state(config2)

for message in state2.values["messages"]:
    print(f"{type(message).__name__}: {message.content}")